# Queen Maintenance — treinamento Qwen2.5-0.5B
Treinamento em GPU no Colab; o modelo final continuará rodando em CPU/RAM no painel.\n

In [ ]:
import torch\nassert torch.cuda.is_available(), 'Ative GPU em Runtime > Change runtime type > T4 GPU'\nprint(torch.cuda.get_device_name(0))\n

In [ ]:
!pip -q install -U transformers datasets accelerate peft trl safetensors sentencepiece\n

In [ ]:
!git clone -q https://github.com/luizgonzntx/pcm-aegea.git\n%cd /content/pcm-aegea\n!mkdir -p /content/data /content/queen_training\n

## Envie os quatro arquivos preparados\n- queen_domain_corpus_v1.jsonl\n- queen_sft_train_v1.jsonl\n- queen_sft_validation_v1.jsonl\n- queen_sft_test_v1.jsonl\n

In [ ]:
from google.colab import files\nimport os, shutil\nuploaded=files.upload()\nfor name in uploaded:\n    shutil.move(name, '/content/data/'+name)\nprint(os.listdir('/content/data'))\n

## Teste curto do pipeline\nPrimeiro usamos 10 mil relatos para validar memória, loss e formato antes do treino integral.\n

In [ ]:
!python training/train_queen_qwen25_05b.py --domain-limit 10000 --output /content/queen_training_smoke\n

## Treino completo\nExecute esta célula somente depois que o teste curto terminar sem erro.\n

In [ ]:
!python training/train_queen_qwen25_05b.py --output /content/queen_training\n

## Avaliação holdout\nGera respostas para exemplos nunca vistos e sinaliza extrapolações óbvias.\n

In [ ]:
!python training/evaluate_queen.py --model /content/queen_training/queen_qwen25_05b_merged --limit 200 --out /content/queen_training/test_predictions.jsonl\n

In [ ]:
import shutil\nshutil.make_archive('/content/queen_training_final','zip','/content/queen_training')\nfiles.download('/content/queen_training_final.zip')\n